<a href="https://colab.research.google.com/github/MariamWassim/internship-/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MariamWassim/internship-/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and

---

why?*

---

**Task type: Classification**

My lane is predicting whether a page is declining and needs a content refresh —
a binary label (`is_declining_label`: 1 = declining, 0 = not). This is
classification, not ranking or scoring, because the target is a discrete
yes/no category, not a continuous number or an ordering. (Ranking on top of
this classifier — like the refresh queue in Notebook 1 — is a downstream step,
not the core prediction task itself.)




## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label`**, derived from `trend_direction == "down"`.

This is a *proxy*, not a directly observed business outcome — it's a rule
defined from the trend bucket, not something a human manually labeled as
"needs a refresh." That matters: the label reflects a specific definition
of decline (a bucketed trend), so my model will only be as good as that
definition. If trend buckets are noisy or coarse, the label inherits that
noise.

In [ ]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df["is_declining_label"].value_counts(normalize=True).round(3))

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50**

Of the top 50 pages my model flags as highest-priority for review, what
fraction are actually declining? I'm choosing this over plain accuracy
because in practice a reviewer only has time to check a limited number of
pages — the metric should match that real constraint. Precision@50 answers
"if I only trust the top 50, how often am I right?" which ties directly to
the content-refresh action a reviewer would take.

In [ ]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print("precision_at_k function defined — will be used once a score exists")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*



**Unit of analysis: one row = one page**, at a single point-in-time snapshot
(not one row per query, or per client). Each page has features like
impressions, average position, CTR, word count, and content age, plus the
derived decline label.

In [ ]:
cols = ["impressions_90d", "avg_position", "ctr", "word_count",
        "content_age_days", "days_since_last_update", "trend_direction",
        "is_declining_label"]
df[cols].head(10)

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (e.g. "stale AND visible" from Notebook 2) only combines a
couple of signals in a hand-picked way. Notebook 2 showed the hand rule
scoring Precision@50 of 0.68 in-sample — already decent — but it can only
express relationships a person thought to write down, and it treats every
threshold as a hard cutoff (stale = 180+ days, visible = 500+ impressions),
which misses pages just under those lines that are still meaningfully
at risk. A learned model can weigh many features together and find
non-linear interactions (e.g. "moderately stale but very high impressions"
vs "very stale but low impressions") that a short if/else can't capture
without becoming unreadable. Notebook 1's full pipeline showed the random
forest reaching Precision@50 ≈ 0.74 vs the rule's 0.24 — directional
evidence that the extra flexibility helps here, even though the honest
caveat from Notebook 2 is that this gap isn't always as large once you
control for tied scores and use a proper holdout.

In [ ]:
import json
res = json.load(open("outputs/model_results.json"))
print("Hand rule Precision@50:", res["baseline"]["baseline_precision_at_50"])
print("Random forest Precision@50:", res["models"]["random_forest"]["precision_at_50"])

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.